In [1]:
from pyspark.sql import SparkSession, DataFrame, Row
from pyspark.sql.functions import col, to_timestamp, unix_timestamp, when, lit, current_timestamp, coalesce, regexp_replace, length, lower
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType
from pyspark.sql.functions import regexp_replace, trim, concat_ws, to_date, least, greatest, current_date, udf

In [2]:
# Initialize Spark session with legacy time parser
spark = SparkSession.builder \
    .appName("BronzeToSilverTransformation") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.eventLog.logBlockUpdates.enabled", True)\
    .enableHiveSupport() \
    .getOrCreate()

In [54]:
ports = spark.sql("SELECT * FROM SILVER.ports")
# spark.sql("SELECT * FROM SILVER.ports")

In [4]:
vessels = spark.sql("SELECT * FROM SILVER.vessels")

In [ ]:
# seismic = spark.sql("SELECT * FROM SILVER.vessels")

In [5]:
spark.sql("use gold")

""


In [56]:
#1. Distinct Vessels Tracked

query = """
CREATE OR REPLACE VIEW gold.vessel_count AS
SELECT COUNT(DISTINCT name) AS distinct_vessel_count
FROM silver.vessels;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.vessel_count")

distinct_vessel_count
82


In [28]:
# 2. Vessel Movements This Week / Month
query = """CREATE OR REPLACE VIEW gold.vessel_movements_summary AS
SELECT
    COUNT(*) AS total_movements,
    COUNT(CASE WHEN arrival_date >= date_trunc('week', current_date) THEN 1 END) AS this_week,
    COUNT(CASE WHEN arrival_date >= date_trunc('month', current_date) THEN 1 END) AS this_month
FROM silver.vessels;
"""

spark.sql(query)
spark.sql("SELECT * FROM gold.vessel_movements_summary")

total_movements,this_week,this_month
444,415,443


In [30]:
# 3. Top 5 Vessel Types by Count
query = """
CREATE OR REPLACE VIEW gold.top_5_vessel_types AS
SELECT
    type,
    COUNT(*) AS vessel_count
FROM silver.vessels
GROUP BY type
ORDER BY vessel_count DESC
LIMIT 5;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.top_5_vessel_types")

type,vessel_count
-,342
Tanker,44
Cargo ship,40
Tanker (HAZ-A),7
Other type,6


In [34]:
# 4. Average Deadweight and Gross Tonnage per Vessel Type
query = """
CREATE OR REPLACE VIEW gold.avg_tonnage_by_vessel_type AS
SELECT
    type,
    AVG(deadweight) AS avg_deadweight,
    AVG(gross_tonnage) AS avg_gross_tonnage
FROM silver.vessels
GROUP BY type;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.avg_tonnage_by_vessel_type")

type,avg_deadweight,avg_gross_tonnage
Tanker,21051.090909090908,11748.34090909091
Other type,3220.0,2042.0
Tanker (HAZ-A),55086.57142857143,30085.14285714286
Cargo ship (HAZ-A),12720.0,10308.0
Tanker (HAZ-B),105778.0,49838.0
-,6066.7397660818715,3657.9152046783624
Cargo ship,25101.775,18655.475


In [39]:
# 5. Vessels Built Before 2000
query = """
CREATE OR REPLACE VIEW gold.vessels_built_before_2000 AS
SELECT
    COUNT(*) AS vessel_count_pre_2000
FROM silver.vessels
WHERE year_built < 2000;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.vessels_built_before_2000")

vessel_count_pre_2000
304


In [41]:
# 6. Vessels per Country (based on destination port country)
query = """
CREATE OR REPLACE VIEW gold.vessels_by_destination_country AS
SELECT
    destination_port_country,
    COUNT(*) AS vessel_count
FROM silver.vessels
GROUP BY destination_port_country;
"""

spark.sql(query)
spark.sql("SELECT * FROM gold.vessels_by_destination_country")

destination_port_country,vessel_count
Yemen,2
Turkey,2
null,84
India,3
Spain,1
Bangladesh,4
-,342
Saudi Arabia,1
Egypt,5


In [42]:
# 7. Vessels per Port in Last 30 Days
query = """
CREATE OR REPLACE VIEW gold.recent_visits_per_port AS
SELECT
    destination_port_name,
    COUNT(*) AS vessel_visits
FROM silver.vessels
WHERE arrival_date >= current_date - INTERVAL 30 DAYS
GROUP BY destination_port_name;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.recent_visits_per_port")

destination_port_name,vessel_visits
Aliaga,1
EGACT,1
Hazira,3
Wadi Feiran,1
RAS SADAT,1
ZET_BAY EG,1
Jeddah,1
El Dekheila,1
Chittagong,1
AIN SUKHNA/EGY,1


In [43]:
# 8. Most Used Vessel Routes
query = """
CREATE OR REPLACE VIEW gold.top_vessel_routes AS
SELECT
    last_port_name,
    destination_port_name,
    COUNT(*) AS route_count
FROM silver.vessels
GROUP BY last_port_name, destination_port_name
ORDER BY route_count DESC;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.top_vessel_routes")

last_port_name,destination_port_name,route_count
-,-,342
As Suways / Suez ...,Destination not a...,18
Alexandria,Destination not a...,13
Port Said,Destination not a...,6
Safaga,SUEZ-SAFAGA,6
Ismailia Anch.,Destination not a...,6
Jeddah Anchorage,JEDDAH<>SAWAKIN,5
Port Tewfik,Destination not a...,5
Singapore Anch. 4,Hazira,3
Chattogram Anch.,Mongla,3


In [45]:
# 9. Average Arrival/Departure Time per Port
query = """
CREATE OR REPLACE VIEW gold.avg_stay_duration_by_port AS
SELECT
    destination_port_name,
    AVG(abs(UNIX_TIMESTAMP(arrival_date) - UNIX_TIMESTAMP(departure_date))) / 3600 AS avg_duration_hours
FROM silver.vessels
WHERE arrival_date IS NOT NULL AND departure_date IS NOT NULL
GROUP BY destination_port_name;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.avg_stay_duration_by_port")

destination_port_name,avg_duration_hours
Aliaga,24.0
EGACT,24.0
Hazira,0.0
Wadi Feiran,600.0
RAS SADAT,0.0
ZET_BAY EG,408.0
Jeddah,408.0
El Dekheila,48.0
Chittagong,576.0
AIN SUKHNA/EGY,96.0


In [47]:
# 10. Vessel Classes by Highest Average Gross Tonnage
query = """
CREATE OR REPLACE VIEW gold.vessel_class_by_avg_tonnage AS
SELECT
    type AS vessel_class,
    AVG(gross_tonnage) AS avg_gross_tonnage
FROM silver.vessels
GROUP BY vessel_class
ORDER BY avg_gross_tonnage DESC;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.vessel_class_by_avg_tonnage")

vessel_class,avg_gross_tonnage
Tanker (HAZ-B),49838.0
Tanker (HAZ-A),30085.14285714286
Cargo ship,18655.475
Tanker,11748.34090909091
Cargo ship (HAZ-A),10308.0
-,3657.9152046783624
Other type,2042.0


In [49]:
# 11. Top 10 Busiest Routes in Last 30 Days

query = """
CREATE OR REPLACE VIEW gold.top_routes_last_30_days AS
SELECT
    last_port_name,
    destination_port_name,
    COUNT(*) AS voyage_count
FROM silver.vessels
WHERE arrival_date >= current_date - INTERVAL 30 DAYS
GROUP BY last_port_name, destination_port_name
ORDER BY voyage_count DESC;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.top_routes_last_30_days limit 10")

last_port_name,destination_port_name,voyage_count
-,-,342
As Suways / Suez ...,Destination not a...,18
Alexandria,Destination not a...,13
Ismailia Anch.,Destination not a...,6
Port Said,Destination not a...,6
Safaga,SUEZ-SAFAGA,6
Jeddah Anchorage,JEDDAH<>SAWAKIN,5
Port Tewfik,Destination not a...,5
Chattogram Anch.,Mongla,3
Singapore Anch. 4,Hazira,3


✅ Gold Views based on silver.ports

In [51]:
# 12. Distinct Ports Count
query = """
CREATE OR REPLACE VIEW gold.distinct_ports_count AS
SELECT COUNT(DISTINCT OID_) AS total_ports
FROM silver.ports;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.distinct_ports_count")
# The output is 0 because all OID_ values are null

total_ports
0


In [52]:
# 13. Ports by Country or Region
query = """
CREATE OR REPLACE VIEW gold.port_distribution_by_region AS
SELECT
    Country_Code,
    Region_Name,
    COUNT(*) AS port_count
FROM silver.ports
GROUP BY Country_Code, Region_Name;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.port_distribution_by_region")

Country_Code,Region_Name,port_count
Canada,Canada Lake Ontar...,44
United Kingdom,England South Coa...,56
Turkey,Turkey Europe -- ...,20
Georgia,Georgia -- 44275,6
Wallis and Futuna,Iles Wallis -- 55640,2
Thailand,Thailand -- 49760,10
U.S. Virgin Islands,St Thomas -- 11305,2
Russia,Russia -- 28670,4
Greece,Greece East Coast...,44
South Georgia and...,South Georgia -- ...,6
